# AMD ROCm Skill Agent 🤔

**目标**: Skill Agent框架实现原理，及性能优化

> 前置: 已完成 `AMD_ROCm_Quick_Start.ipynb` 的环境验证，最好熟悉了 `AMD_ROCm_LLM_Inference.ipynb`的大模型推理原理。

## Chapter 1: Skill Agent Framework

### Design Philosophy

Skill Agent is a modular AI agent architecture. Core idea: decompose agent capabilities into composable **Skill** units:

```
+---------------------------------------------+
|              Skill Agent                     |
|  +-----------+  +-----------+               |
|  | Skill A   |  | Skill B   |  ...          |
|  | (search)  |  | (compute) |               |
|  +-----+-----+  +-----+-----+               |
|        |               |                     |
|  +-----+---------------+------------------+ |
|  |         LLM Backend                    | |
|  |    (ROCm GPU Inference)                | |
|  +----------------------------------------+ |
+---------------------------------------------+
```

This chapter builds an extensible Skill Agent framework targeting AMD GPU inference.

In [ ]:
# ============================================================
# Cell 1.1: Skill Base Class
# All specialized skills inherit from this abstract base
# ============================================================

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import json, time


@dataclass
class SkillResult:
    """Skill execution result wrapper"""
    success: bool
    data: Any = None
    error: str = ""
    elapsed_ms: float = 0.0


class Skill(ABC):
    """Base Skill class -- all specialized skills inherit from this
    
    Each Skill must implement:
    - name: skill identifier
    - description: what it does (used by LLM for auto-selection)
    - execute(): core execution logic
    """
    
    @property
    @abstractmethod
    def name(self) -> str:
        """Skill name"""
        pass
    
    @property
    @abstractmethod
    def description(self) -> str:
        """Skill description for agent auto-selection"""
        pass
    
    @property
    def parameters_schema(self) -> Dict:
        """Parameter schema in JSON Schema format"""
        return {}
    
    @abstractmethod
    def execute(self, **kwargs) -> SkillResult:
        """Execute skill, must be implemented by subclass"""
        pass
    
    def __call__(self, **kwargs) -> SkillResult:
        """Direct call with auto-timing"""
        t0 = time.time()
        try:
            result = self.execute(**kwargs)
            result.elapsed_ms = (time.time() - t0) * 1000
            return result
        except Exception as e:
            return SkillResult(
                success=False,
                error=f"{self.name}: {str(e)}",
                elapsed_ms=(time.time() - t0) * 1000
            )
    
    def to_prompt(self) -> str:
        """Generate skill description for LLM prompt"""
        return f"- {self.name}: {self.description}"


# ============================================================
# Example Skill 1: Calculator
# ============================================================
class CalculatorSkill(Skill):
    """Math calculation -- evaluates expressions safely"""
    
    @property
    def name(self) -> str:
        return "calculator"
    
    @property
    def description(self) -> str:
        return "Evaluate math expressions, e.g. '2+3**4' or 'sqrt(16)'"
    
    @property
    def parameters_schema(self) -> Dict:
        return {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression"}
            },
            "required": ["expression"]
        }
    
    def execute(self, expression: str = "", **kwargs) -> SkillResult:
        import math
        allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        allowed_names["abs"] = abs
        allowed_names["round"] = round
        
        try:
            result = eval(expression, {"__builtins__": {}}, allowed_names)
            return SkillResult(success=True, data={"expression": expression, "result": result})
        except Exception as e:
            return SkillResult(success=False, error=f"Expression error: {e}")


# ============================================================
# Example Skill 2: Text Processor
# ============================================================
class TextProcessorSkill(Skill):
    """Text processing -- stats, keywords, summarization"""
    
    @property
    def name(self) -> str:
        return "text_processor"
    
    @property
    def description(self) -> str:
        return "Text statistics, keyword extraction, summarization."
    
    @property
    def parameters_schema(self) -> Dict:
        return {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Input text"},
                "operation": {
                    "type": "string",
                    "enum": ["stats", "keywords", "summarize"],
                    "description": "Operation type"
                }
            },
            "required": ["text", "operation"]
        }
    
    def execute(self, text: str = "", operation: str = "stats", **kwargs) -> SkillResult:
        if operation == "stats":
            words = text.split()
            chars = len(text)
            lines = text.count("\n") + 1
            return SkillResult(success=True, data={
                "words": len(words), "characters": chars, "lines": lines,
                "avg_word_len": sum(len(w) for w in words) / max(len(words), 1)
            })
        elif operation == "keywords":
            from collections import Counter
            import re
            words = re.findall(r'\w+', text.lower())
            words = [w for w in words if len(w) > 1]
            counter = Counter(words)
            return SkillResult(success=True, data={"top_keywords": counter.most_common(10)})
        else:
            summary = text[:200] + "..." if len(text) > 200 else text
            return SkillResult(success=True, data={"summary": summary})


# ============================================================
# Quick test
# ============================================================
print("[TEST] Skill base classes...")
calc = CalculatorSkill()
result = calc(expression="2**10 + sqrt(144)")
print(f"   Calculator: 2**10 + sqrt(144) = {result.data['result']} ({result.elapsed_ms:.2f}ms)")

tproc = TextProcessorSkill()
result = tproc(text="AMD GPU ROCm AI inference optimization deep learning transformer",
               operation="keywords")
print(f"   TextProcessor keywords: {result.data['top_keywords']}")
print("[OK] All Skill base tests passed")

[TEST] Skill base classes...
   Calculator: 2**10 + sqrt(144) = 1036.0 (0.04ms)
   TextProcessor keywords: [('amd', 1), ('gpu', 1), ('rocm', 1), ('ai', 1), ('inference', 1), ('optimization', 1), ('deep', 1), ('learning', 1), ('transformer', 1)]
[OK] All Skill base tests passed


In [54]:
# ============================================================
# Cell 1.2: SkillManager -- Skill Registry & Router
# ============================================================

class SkillManager:
    """Skill Manager: register, discover, and execute skills"""
    
    def __init__(self):
        self._skills: Dict[str, Skill] = {}
    
    def register(self, skill: Skill) -> None:
        """Register a skill"""
        self._skills[skill.name] = skill
        print(f"   [REG] {skill.name}: {skill.description[:40]}...")
    
    def register_many(self, skills: List[Skill]) -> None:
        """Batch register"""
        for skill in skills:
            self.register(skill)
    
    def get(self, name: str) -> Optional[Skill]:
        """Get skill by name"""
        return self._skills.get(name)
    
    def list_skills(self) -> List[Dict]:
        """List all registered skills"""
        return [
            {"name": s.name, "description": s.description, "parameters": s.parameters_schema}
            for s in self._skills.values()
        ]
    
    def execute(self, skill_name: str, **kwargs) -> SkillResult:
        """Execute skill by name"""
        skill = self.get(skill_name)
        if not skill:
            return SkillResult(
                success=False,
                error=f"Unknown skill: {skill_name}. Available: {list(self._skills.keys())}"
            )
        return skill(**kwargs)
    
    def build_system_prompt(self) -> str:
        """Build LLM system prompt with all available skills"""
        skills_text = "\n".join(
            f"- {s.name}: {s.description} {s.parameters_schema}" for s in self._skills.values()
        )
        return f"""You are a skill assistant. Available tools:

{skills_text}

When using a tool, respond with JSON:
{{"tool": "tool_name", "parameters": {{"param1": "value1"}}}}

If no tool needed, answer directly."""
    
    def __len__(self) -> int:
        return len(self._skills)
    
    def __repr__(self) -> str:
        return f"SkillManager({len(self._skills)} skills: {list(self._skills.keys())})"


# Initialize
skill_manager = SkillManager()
skill_manager.register(CalculatorSkill())
skill_manager.register(TextProcessorSkill())

print(f"\n[INFO] {skill_manager}")
print(f"\n[INFO] Skills JSON:\n{json.dumps(skill_manager.list_skills(), indent=2, ensure_ascii=False)}")
print(f"\n[INFO] System Prompt preview:\n{skill_manager.build_system_prompt()[:400]}...")

   [REG] calculator: Evaluate math expressions, e.g. '2+3**4'...
   [REG] text_processor: Text statistics, keyword extraction, sum...

[INFO] SkillManager(2 skills: ['calculator', 'text_processor'])

[INFO] Skills JSON:
[
  {
    "name": "calculator",
    "description": "Evaluate math expressions, e.g. '2+3**4' or 'sqrt(16)'",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "Math expression"
        }
      },
      "required": [
        "expression"
      ]
    }
  },
  {
    "name": "text_processor",
    "description": "Text statistics, keyword extraction, summarization, Don't do other tasks!",
    "parameters": {
      "type": "object",
      "properties": {
        "text": {
          "type": "string",
          "description": "Input text"
        },
        "operation": {
          "type": "string",
          "enum": [
            "stats",
            "keywords",
            "summarize"


In [55]:
# ============================================================
# Cell 1.3: LLM Backend -- ROCm GPU Inference Support
# ============================================================

class LLMBackend:
    """LLM inference backend with AMD GPU (ROCm) support + CPU fallback
    
    In a real ROCm environment, uses HuggingFace Transformers on GPU.
    This demo provides a rule-based fallback for any environment.
    """
    
    def __init__(self, use_gpu: bool = True):
        import torch
        self.device = torch.device("cuda" if torch.cuda.is_available() and use_gpu else "cpu")
        self.device_name = torch.cuda.get_device_name(0) if self.device.type == "cuda" else "CPU"
        self.model = None
        self.tokenizer = None
        print(f"   LLM Backend device: {self.device_name} ({self.device})")
    
    def load_model(self, model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"):
        """Load HuggingFace model to ROCm GPU (or CPU fallback)
        
        On ROCm:
            model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=torch.float16, device_map="auto"
            ).to(device)
        """
        try:
            # from transformers import AutoModelForCausalLM, AutoTokenizer
            from modelscope import AutoModelForCausalLM, AutoTokenizer
            import torch
            
            print(f"   Loading model: {model_name} ...")
            
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            dtype = torch.float16 if self.device.type == "cuda" else torch.float32
            
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=dtype,
                device_map="auto" if self.device.type == "cuda" else None,
                trust_remote_code=True,
            )
            
            if self.device.type != "cuda":
                self.model = self.model.to(self.device)
            
            params = sum(p.numel() for p in self.model.parameters())
            print(f"   [OK] Model loaded: {params/1e9:.2f}B params -> {self.device_name}")
            return True
            
        except ImportError:
            print(f"   [WARN] transformers not installed, using demo mode")
            return False
        except Exception as e:
            print(f"   [WARN] Model load failed ({e}), using demo mode")
            return False
    
    def generate(self, prompt: str, max_tokens: int = 256, temperature: float = 0.7) -> str:
        """Generate text using ROCm GPU (or CPU/demo fallback)"""
        if self.model is not None and self.tokenizer is not None:
            import torch
            messages = [{"role": "user", "content": prompt}]
            text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
            
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs, max_new_tokens=max_tokens, temperature=temperature,
                    do_sample=temperature > 0, pad_token_id=self.tokenizer.pad_token_id,
                )
            
            response = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
            return response.strip()
        else:
            return self._demo_generate(prompt)
    
    def _demo_generate(self, prompt: str) -> str:
        """Demo mode: rule-based response (no GPU needed)"""
        prompt_lower = prompt.lower()
        
        if any(w in prompt_lower for w in ["calc", "compute", "+", "*", "-", "/"]):
            return '{"tool": "calculator", "parameters": {"expression": "2**10 + sqrt(144)"}}'
        elif any(w in prompt_lower for w in ["stats", "keyword", "text"]):
            return '{"tool": "text_processor", "parameters": {"text": "AMD GPU ROCm AI inference", "operation": "stats"}}'
        else:
            return "I am a Skill Agent powered by AMD GPU. I can help you with various tasks using my skills."


# Test LLM Backend
llm = LLMBackend(use_gpu=True)
# Optional: load a real model
llm.load_model("qwen/Qwen2.5-0.5B-Instruct")

test_prompt = "Calculate 2^10 + sqrt(144)"
response = llm.generate(test_prompt)
print(f"\nPrompt: {test_prompt}")
print(f"Response: {response}")


   LLM Backend device:  (cuda)
   Loading model: qwen/Qwen2.5-0.5B-Instruct ...


2026-05-31 10:34:02,581 - modelscope - INFO - Target directory already exists, skipping creation.


2026-05-31 10:34:03,700 - modelscope - INFO - Target directory already exists, skipping creation.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2050.38it/s]


   [OK] Model loaded: 0.49B params -> 

Prompt: Calculate 2^10 + sqrt(144)
Response: To calculate \(2^{10} + \sqrt{144}\), we will break it down into two main steps: first, calculating \(2^{10}\), and second, calculating \(\sqrt{144}\).

### Step 1: Calculate \(2^{10}\)
The expression \(2^{10}\) means that we multiply 2 by itself 10 times:
\[
2^{10} = 2 \times 2 \times 2 \times 2 \times 2 \times 2 \times 2 \times 2 \times 2 \times 2
\]

Breaking this down step-by-step:
\[
2 \times 2 = 4
\]
\[
4 \times 2 = 8
\]
\[
8 \times 2 = 16
\]
\[
16 \times 2 = 32
\]
\[
32 \times 2 = 64
\]
\[
64 \times 2 = 128
\]

So, \(2^{10} = 128\).

### Step 2: Calculate \(\sqrt{144}\


In [ ]:
# ============================================================
# Cell 1.4: SkillAgent -- Complete Agent Loop
# LLM decides which skill to call -> execute skill -> return result
# ============================================================

import json, re

class SkillAgent:
    """Skill Agent: composes LLM Backend + Skill Manager into a complete agent
    
    Workflow:
    1. Receive user input
    2. LLM decides which skill to call
    3. Execute skill
    4. Return result
    """
    
    def __init__(self, llm_backend: LLMBackend, skill_manager: SkillManager, max_rounds: int = 3):
        self.llm = llm_backend
        self.skills = skill_manager
        self.max_rounds = max_rounds
        self.history: List[Dict] = []
    
    def run(self, user_input: str) -> str:
        """Execute one complete agent conversation turn"""
        
        print(f"\n{'='*50}")
        print(f"[USER] {user_input}")
        
        # Step 1: Build prompt
        system_prompt = self.skills.build_system_prompt()
        full_prompt = f"{system_prompt}\n\nUser request: {user_input}\n\nRespond (or return JSON tool call):"
        print(f"\n\n[PROMPT_START]: \n{full_prompt}\n[PROMPT_END]\n\n")
        
        # Step 2: LLM inference (ROCm GPU)
        llm_response = self.llm.generate(full_prompt)
        print(f"[LLM] {llm_response[:200]}...")
        
        # Step 3: Parse tool call
        tool_call = self._parse_tool_call(llm_response)
        if tool_call:
            tool_name = tool_call.get("tool")
            params = tool_call.get("parameters", {})
            print(f"[TOOL] Calling: {tool_name}({params})")
            
            # Step 4: Execute skill
            result = self.skills.execute(tool_name, **params)
            
            if result.success:
                response = f"[OK] Skill [{tool_name}] success:\n{json.dumps(result.data, indent=2, ensure_ascii=False)}\nTime: {result.elapsed_ms:.2f}ms"
            else:
                response = f"[ERR] Skill failed: {result.error}"
        else:
            response = llm_response
        
        print(f"[AGENT] {response[:200]}...")
        print(f"{'='*50}")
        
        self.history.append({"user": user_input, "response": response})
        return response
    
    def _parse_tool_call(self, text: str) -> Optional[Dict]:
        """Extract JSON tool call from LLM output"""
        try:
            return json.loads(text)
        except:
            return None


# ============================================================
# Full Test
# ============================================================
print("\n" + "=" * 60)
print("[TEST] SkillAgent Full Test")
print("=" * 60)

agent = SkillAgent(llm, skill_manager)

# Test 1: Calculator
print("\n--- Test 1: Math Calculation ---")
agent.run("Calculate 2**10 + sqrt(144)")

# Test 2: Text Stats
print("\n--- Test 2: Text Statistics ---")
agent.run("Get stats for: AMD GPU ROCm AI inference optimization")

# Test 3: 非skill类问题 (很容易误判-升级大模型)
print("\n--- Test 3: Ramdom Statistics ---")
agent.run("爸爸的姐姐叫什么")

print("\n[OK] SkillAgent framework test complete!")


[TEST] SkillAgent Full Test

--- Test 1: Math Calculation ---

[USER] Calculate 2**10 + sqrt(144)


[PROMPT_START]: 
You are a skill assistant. Available tools:

- calculator: Evaluate math expressions, e.g. '2+3**4' or 'sqrt(16)' {'type': 'object', 'properties': {'expression': {'type': 'string', 'description': 'Math expression'}}, 'required': ['expression']}
- text_processor: Text statistics, keyword extraction, summarization, Don't do other tasks! {'type': 'object', 'properties': {'text': {'type': 'string', 'description': 'Input text'}, 'operation': {'type': 'string', 'enum': ['stats', 'keywords', 'summarize'], 'description': 'Operation type'}}, 'required': ['text', 'operation']}

When using a tool, respond with JSON:
{"tool": "tool_name", "parameters": {"param1": "value1"}}

If no tool needed, answer directly.

User request: Calculate 2**10 + sqrt(144)

Respond (or return JSON tool call):
[PROMPT_END]


[LLM] {"tool": "calculator", "parameters": {"expression": "2**10 + sqrt(144)"}}

## Chapter 2: Inference Optimization on AMD GPU

AMD GPU (ROCm) inference optimization techniques:

| Technique | Principle | Speedup | Precision Loss | ROCm Support |
|-----------|-----------|---------|---------------|-------------|
| **FP16/BF16** | Half precision | ~2x | Minimal | Native |
| **INT8 Quant** | 8-bit integer | ~4x | <1% | bitsandbytes |
| **INT4 Quant** | 4-bit integer | ~6x | 1-3% | bitsandbytes |
| **KV-Cache** | Cache Key-Value | ~2x (decode) | None | Built-in |
| **Flash Attn** | IO-aware attention | ~2-3x | Numerically eq. | ROCm 5.7+ |
| **Cont. Batching** | Dynamic batching | ~5-10x (throughput) | None | vLLM |
| **Tensor Parallel** | Multi-GPU parallel | ~Nx | None | PyTorch FSDP |

This chapter demonstrates key optimization techniques. Code logic directly migrates to real ROCm environments.

In [60]:
# ============================================================
# Cell 2.1: Baseline Inference Benchmark
# Simulates 7B model transformer inference on AMD GPU
# ============================================================

import torch, time, numpy as np

print("=" * 60)
print("[BENCH] Baseline Inference Performance")
print("=" * 60)

# Simulated 7B model config
BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, NUM_LAYERS = 1, 1024, 4096, 32
DEVICE_BENCH = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def benchmark_matmul(device, size=HIDDEN_DIM, seq_len=SEQ_LEN, warmup=5, iters=20):
    """Benchmark matrix multiplication -- core op in LLM inference"""
    x = torch.randn(seq_len, size, device=device)
    w = torch.randn(size, size, device=device)
    
    # Warmup (ROCm: first call triggers kernel compilation)
    for _ in range(warmup):
        _ = torch.matmul(x, w)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = torch.matmul(x, w)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    elapsed = (time.perf_counter() - t0) / iters * 1000  # ms
    flops = 2 * seq_len * size * size
    tflops = flops / (elapsed / 1000) / 1e12
    
    return elapsed, tflops


def benchmark_attention(device, batch=BATCH_SIZE, seq_len=SEQ_LEN,
                       dim=HIDDEN_DIM, heads=32, warmup=5, iters=20):
    """Benchmark self-attention performance"""
    head_dim = dim // heads
    q = torch.randn(batch, heads, seq_len, head_dim, device=device)
    k = torch.randn(batch, heads, seq_len, head_dim, device=device)
    v = torch.randn(batch, heads, seq_len, head_dim, device=device)
    
    for _ in range(warmup):
        attn = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        _ = torch.matmul(attn, v)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    t0 = time.perf_counter()
    for _ in range(iters):
        attn = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        _ = torch.matmul(attn, v)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    elapsed = (time.perf_counter() - t0) / iters * 1000
    flops = (2 * batch * heads * seq_len * seq_len * head_dim) * 2
    tflops = flops / (elapsed / 1000) / 1e12
    
    return elapsed, tflops


# Run benchmarks
print(f"\nTest Device: {DEVICE_BENCH}")
if DEVICE_BENCH.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print(f"\nConfig: 7B model, hidden={HIDDEN_DIM}, layers={NUM_LAYERS}, seq={SEQ_LEN}")

# MatMul benchmark
print(f"\n--- MatMul Benchmark ---")
matmul_ms, matmul_tflops = benchmark_matmul(DEVICE_BENCH, HIDDEN_DIM, SEQ_LEN)
print(f"   Single layer MatMul: {matmul_ms:.3f} ms")
print(f"   Throughput: {matmul_tflops:.2f} TFLOPS")
print(f"   Est. 32 layers: {matmul_ms * NUM_LAYERS:.1f} ms/token (MatMul only)")

# Attention benchmark
print(f"\n--- Attention Benchmark ---")
attn_ms, attn_tflops = benchmark_attention(DEVICE_BENCH)
print(f"   Single layer Attention: {attn_ms:.3f} ms")
print(f"   Attention throughput: {attn_tflops:.2f} TFLOPS")

# Summary
total_per_layer = matmul_ms + attn_ms
total_32_layers = total_per_layer * NUM_LAYERS
print(f"\n--- Summary ---")
print(f"   Per-layer total: {total_per_layer:.3f} ms")
print(f"   32-layer total: {total_32_layers:.1f} ms/token")
print(f"   Est. throughput: {1000/total_32_layers:.1f} tokens/s")

BASELINE = {
    "matmul_ms": matmul_ms, "attn_ms": attn_ms,
    "total_per_layer": total_per_layer,
    "tokens_per_sec": 1000 / total_32_layers
}
print(f"\n[SAVED] Baseline: {json.dumps({k: round(v, 2) for k, v in BASELINE.items()}, indent=2)}")

[BENCH] Baseline Inference Performance

Test Device: cuda
   GPU: 
   VRAM: 191.7 GB

Config: 7B model, hidden=4096, layers=32, seq=1024

--- MatMul Benchmark ---
   Single layer MatMul: 1.245 ms
   Throughput: 27.60 TFLOPS
   Est. 32 layers: 39.8 ms/token (MatMul only)

--- Attention Benchmark ---
   Single layer Attention: 1.246 ms
   Attention throughput: 13.79 TFLOPS

--- Summary ---
   Per-layer total: 2.490 ms
   32-layer total: 79.7 ms/token
   Est. throughput: 12.5 tokens/s

[SAVED] Baseline: {
  "matmul_ms": 1.24,
  "attn_ms": 1.25,
  "total_per_layer": 2.49,
  "tokens_per_sec": 12.55
}


In [61]:
# ============================================================
# Cell 2.2: FP16/BF16 Mixed Precision Inference
# AMD GPU (ROCm) has native FP16 hardware support
# MI300X FP16 compute is 8x FP32 throughput
# ============================================================

print("=" * 60)
print("[OPT] FP16/BF16 Mixed Precision Inference")
print("=" * 60)

def compare_precision(batch=1, seq_len=SEQ_LEN, hidden=HIDDEN_DIM, iters=50):
    """Compare inference performance across precisions"""
    results = {}
    
    x = torch.randn(seq_len, hidden)
    w = torch.randn(hidden, hidden)
    
    precisions = [
        ("FP32", torch.float32, "Baseline precision"),
        ("FP16", torch.float16, "Half precision, ~2x speedup"),
        ("BF16", torch.bfloat16, "Brain Float, larger range"),
    ]
    
    for name, dtype, desc in precisions:
        try:
            x_d = x.to(DEVICE_BENCH).to(dtype)
            w_d = w.to(DEVICE_BENCH).to(dtype)
            
            for _ in range(10):
                _ = torch.matmul(x_d, w_d)
            if DEVICE_BENCH.type == "cuda":
                torch.cuda.synchronize()
            
            t0 = time.perf_counter()
            for _ in range(iters):
                _ = torch.matmul(x_d, w_d)
            if DEVICE_BENCH.type == "cuda":
                torch.cuda.synchronize()
            
            elapsed_ms = (time.perf_counter() - t0) / iters * 1000
            speedup = BASELINE["matmul_ms"] / elapsed_ms
            
            ref = torch.matmul(x.to(DEVICE_BENCH), w.to(DEVICE_BENCH))
            test = torch.matmul(x_d, w_d).float()
            mae = (ref - test).abs().mean().item()
            
            results[name] = {"time_ms": elapsed_ms, "speedup": speedup, "mae": mae, "desc": desc}
            print(f"\n   {name} ({desc}):")
            print(f"      Time: {elapsed_ms:.4f} ms")
            print(f"      Speedup: {speedup:.2f}x vs FP32")
            print(f"      MAE: {mae:.6f}")
            
        except Exception as e:
            print(f"   {name}: Not supported ({e})")
    
    return results

precision_results = compare_precision()

print(f"\n[SUMMARY] Precision Comparison:")
for name, data in precision_results.items():
    print(f"   {name}: {data['speedup']:.2f}x speedup, MAE={data['mae']:.6f}")

print(f"\n[ROCm Best Practice]:")
print(f"   - Inference: prefer FP16 (AMD GPU native hardware accel)")
print(f"   - Training: prefer BF16 (better numerical stability)")
print(f"   - PyTorch ROCm: model.half() or torch_dtype=torch.float16")

[OPT] FP16/BF16 Mixed Precision Inference

   FP32 (Baseline precision):
      Time: 1.2432 ms
      Speedup: 1.00x vs FP32
      MAE: 0.000000

   FP16 (Half precision, ~2x speedup):
      Time: 0.1872 ms
      Speedup: 6.65x vs FP32
      MAE: 0.018120

   BF16 (Brain Float, larger range):
      Time: 0.1890 ms
      Speedup: 6.59x vs FP32
      MAE: 0.144985

[SUMMARY] Precision Comparison:
   FP32: 1.00x speedup, MAE=0.000000
   FP16: 6.65x speedup, MAE=0.018120
   BF16: 6.59x speedup, MAE=0.144985

[ROCm Best Practice]:
   - Inference: prefer FP16 (AMD GPU native hardware accel)
   - Training: prefer BF16 (better numerical stability)
   - PyTorch ROCm: model.half() or torch_dtype=torch.float16


In [62]:
# ============================================================
# Cell 2.3: INT8/INT4 Quantized Inference
# AMD GPU (ROCm) quantization: bitsandbytes (ROCm 5.4+)
# ============================================================

print("=" * 60)
print("[OPT] INT8/INT4 Quantized Inference")
print("=" * 60)

print("AMD GPU (ROCm) quantization options:")
print("   - bitsandbytes: LLM.int8(), NF4 (ROCm 5.4+)")
print("   - torch.ao.quantization: PyTorch native")
print("   - llama.cpp: GGUF format")
print("   - vLLM: AWQ/GPTQ + ROCm support")


def simulate_weight_quantization(dim=HIDDEN_DIM, bits=8):
    """Simulate weight quantization to demonstrate the concept"""
    
    fp32_weights = torch.randn(dim, dim)
    max_val = fp32_weights.abs().max().item()
    
    if bits == 8:
        scale = max_val / 127.0
        quantized = torch.clamp(torch.round(fp32_weights / scale), -128, 127).to(torch.int8)
        decompressed = quantized.float() * scale
        memory_saved = (1 - 1/4) * 100  # FP32(4B) -> INT8(1B)
    elif bits == 4:
        scale = max_val / 7.0
        quantized = torch.clamp(torch.round(fp32_weights / scale), -8, 7).to(torch.int8)
        decompressed = quantized.float() * scale
        memory_saved = (1 - 1/8) * 100  # FP32(4B) -> INT4(0.5B)
    else:
        raise ValueError(f"Unsupported bit width: {bits}")
    
    mae = (fp32_weights - decompressed).abs().mean().item()
    rmse = torch.sqrt(((fp32_weights - decompressed) ** 2).mean()).item()
    
    return {"bits": bits, "memory_saved_pct": memory_saved, "mae": mae, "rmse": rmse, "scale": scale}


# INT8 quantization
print(f"\n--- INT8 Weight Quantization ---")
int8_result = simulate_weight_quantization(HIDDEN_DIM, bits=8)
print(f"   VRAM saved: {int8_result['memory_saved_pct']:.0f}%")
print(f"   MAE: {int8_result['mae']:.8f}")
print(f"   RMSE: {int8_result['rmse']:.8f}")

# INT4 quantization
print(f"\n--- INT4 Weight Quantization (NF4 style) ---")
int4_result = simulate_weight_quantization(HIDDEN_DIM, bits=4)
print(f"   VRAM saved: {int4_result['memory_saved_pct']:.0f}%")
print(f"   MAE: {int4_result['mae']:.8f}")
print(f"   RMSE: {int4_result['rmse']:.8f}")

# Summary table
print(f"\n[TABLE] Quantization Comparison:")
print(f"   {'Scheme':<12} {'VRAM Saved':>12} {'Est.Speedup':>14} {'Precision Loss':>16}")
print(f"   {'-'*56}")
print(f"   {'FP32 Baseline':<12} {'0%':>12} {'1.0x':>14} {'0':>16}")
print(f"   {'FP16':<12} {'50%':>12} {'~2x':>14} {'<0.01%':>16}")
print(f"   {'INT8':<12} {'75%':>12} {'~4x':>14} {'<0.5%':>16}")
print(f"   {'INT4':<12} {'87.5%':>12} {'~6x':>14} {'1-3%':>16}")

print(f"\n[ROCm Quantization Code]:")
print(f"   # bitsandbytes INT8")
print(f"   from transformers import BitsAndBytesConfig")
print(f"   bnb_config = BitsAndBytesConfig(load_in_8bit=True)")
print(f"   model = AutoModel.from_pretrained('model', quantization_config=bnb_config)")
print(f"")
print(f"   # bitsandbytes NF4 (recommended)")
print(f"   bnb_config = BitsAndBytesConfig(")
print(f"       load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,")
print(f"       bnb_4bit_quant_type='nf4'")
print(f"   )")

[OPT] INT8/INT4 Quantized Inference
AMD GPU (ROCm) quantization options:
   - bitsandbytes: LLM.int8(), NF4 (ROCm 5.4+)
   - torch.ao.quantization: PyTorch native
   - llama.cpp: GGUF format
   - vLLM: AWQ/GPTQ + ROCm support

--- INT8 Weight Quantization ---
   VRAM saved: 75%
   MAE: 0.01012965
   RMSE: 0.01169607

--- INT4 Weight Quantization (NF4 style) ---
   VRAM saved: 88%
   MAE: 0.18840247
   RMSE: 0.21756029

[TABLE] Quantization Comparison:
   Scheme         VRAM Saved    Est.Speedup   Precision Loss
   --------------------------------------------------------
   FP32 Baseline           0%           1.0x                0
   FP16                  50%            ~2x           <0.01%
   INT8                  75%            ~4x            <0.5%
   INT4                87.5%            ~6x             1-3%

[ROCm Quantization Code]:
   # bitsandbytes INT8
   from transformers import BitsAndBytesConfig
   bnb_config = BitsAndBytesConfig(load_in_8bit=True)
   model = AutoModel.from_p

In [63]:
# ============================================================
# Cell 2.4: KV-Cache Optimization
# For autoregressive generation, cache historical K/V matrices
# Reduces compute from O(n^2) to O(n) for n tokens
# ============================================================

print("=" * 60)
print("[OPT] KV-Cache Optimization")
print("=" * 60)

class KVCache:
    """Key-Value Cache for efficient autoregressive generation"""
    
    def __init__(self, max_batch=1, max_seq_len=2048, num_heads=32, head_dim=128):
        self.max_seq_len = max_seq_len
        self.num_heads = num_heads
        self.head_dim = head_dim
        
        self.k_cache = torch.zeros(max_batch, num_heads, max_seq_len, head_dim)
        self.v_cache = torch.zeros(max_batch, num_heads, max_seq_len, head_dim)
        self.length = 0
    
    def update(self, k_new: torch.Tensor, v_new: torch.Tensor):
        """Append new K/V tensors"""
        seq_len = k_new.shape[2]
        
        if self.length + seq_len > self.max_seq_len:
            shift = self.length + seq_len - self.max_seq_len
            self.k_cache[:, :, :self.length - shift] = self.k_cache[:, :, shift:self.length]
            self.v_cache[:, :, :self.length - shift] = self.v_cache[:, :, shift:self.length]
            self.length -= shift
        
        self.k_cache[:, :, self.length:self.length + seq_len] = k_new
        self.v_cache[:, :, self.length:self.length + seq_len] = v_new
        self.length += seq_len
    
    def get_kv(self):
        """Get full K/V cache"""
        return self.k_cache[:, :, :self.length], self.v_cache[:, :, :self.length]
    
    def clear(self):
        self.length = 0
    
    @property
    def memory_mb(self):
        """Cache VRAM usage (MB)"""
        return self.k_cache.element_size() * self.k_cache.nelement() / 1024 / 1024 * 2


# Performance comparison: with vs without KV-Cache
def simulate_generation_kv_cache(num_tokens=100, seq_len=256, hidden=1024, heads=16):
    """Simulate autoregressive generation, compare with/without KV-Cache"""
    
    head_dim = hidden // heads
    cache = KVCache(max_seq_len=seq_len + num_tokens, num_heads=heads, head_dim=head_dim)
    
    k_prompt = torch.randn(1, heads, seq_len, head_dim)
    v_prompt = torch.randn(1, heads, seq_len, head_dim)
    cache.update(k_prompt, v_prompt)
    
    times_with_cache, times_without_cache = [], []
    
    for i in range(num_tokens):
        cur_len = seq_len + i
        
        t0 = time.perf_counter()
        k_new = torch.randn(1, heads, 1, head_dim)
        v_new = torch.randn(1, heads, 1, head_dim)
        cache.update(k_new, v_new)
        k_full, v_full = cache.get_kv()
        _ = torch.matmul(k_new, k_full.transpose(-2, -1))
        times_with_cache.append(time.perf_counter() - t0)
        
        t0 = time.perf_counter()
        k_all = torch.randn(1, heads, cur_len + 1, head_dim)
        v_all = torch.randn(1, heads, cur_len + 1, head_dim)
        _ = torch.matmul(k_all[:, :, -1:, :], k_all.transpose(-2, -1))
        times_without_cache.append(time.perf_counter() - t0)
    
    avg_with = np.mean(times_with_cache) * 1000
    avg_without = np.mean(times_without_cache) * 1000
    return avg_with, avg_without

print(f"\n--- KV-Cache Performance Comparison ---")
avg_with, avg_without = simulate_generation_kv_cache(num_tokens=50)
speedup = avg_without / avg_with
print(f"   With KV-Cache avg: {avg_with:.4f} ms/token")
print(f"   Without KV-Cache avg: {avg_without:.4f} ms/token")
print(f"   Speedup: {speedup:.2f}x")

print(f"\n--- KV-Cache VRAM Analysis (7B model, seq=4096) ---")
kv_cache = KVCache(max_seq_len=4096, num_heads=32, head_dim=128)
print(f"   Single layer KV-Cache: {kv_cache.memory_mb:.1f} MB")
print(f"   32-layer KV-Cache: {kv_cache.memory_mb * 32:.1f} MB")
print(f"   Weights (FP16): ~14 GB")
print(f"   Total VRAM (with KV-Cache): ~{14 + kv_cache.memory_mb * 32 / 1024:.1f} GB")
print(f"   MI300X (192GB) usage: {(14 + kv_cache.memory_mb * 32 / 1024) / 192 * 100:.1f}%")

print(f"\n[Best Practice]:")
print(f"   - HuggingFace: use_cache=True (default)")
print(f"   - vLLM: PagedAttention for better VRAM fragmentation")
print(f"   - Long context: consider GQA/MQA to reduce KV heads")

[OPT] KV-Cache Optimization

--- KV-Cache Performance Comparison ---
   With KV-Cache avg: 0.3162 ms/token
   Without KV-Cache avg: 2.1767 ms/token
   Speedup: 6.88x

--- KV-Cache VRAM Analysis (7B model, seq=4096) ---
   Single layer KV-Cache: 128.0 MB
   32-layer KV-Cache: 4096.0 MB
   Weights (FP16): ~14 GB
   Total VRAM (with KV-Cache): ~18.0 GB
   MI300X (192GB) usage: 9.4%

[Best Practice]:
   - HuggingFace: use_cache=True (default)
   - vLLM: PagedAttention for better VRAM fragmentation
   - Long context: consider GQA/MQA to reduce KV heads


In [64]:
# ============================================================
# Cell 2.5: Batch Inference & Throughput Optimization
# ============================================================

print("=" * 60)
print("[OPT] Batch Inference Throughput")
print("=" * 60)

def benchmark_batch_inference(batch_sizes=[1, 2, 4, 8, 16],
                            seq_len=512, hidden=4096, iters=30):
    """Test throughput at different batch sizes"""
    
    results = []
    device = DEVICE_BENCH
    
    for bs in batch_sizes:
        try:
            x = torch.randn(bs, seq_len, hidden, device=device)
            w1 = torch.randn(hidden, hidden, device=device)
            w2 = torch.randn(hidden, hidden, device=device)
            
            for _ in range(10):
                h = torch.matmul(x, w1)
                h = torch.nn.functional.relu(h)
                _ = torch.matmul(h, w2)
            if device.type == "cuda":
                torch.cuda.synchronize()
            
            t0 = time.perf_counter()
            for _ in range(iters):
                h = torch.matmul(x, w1)
                h = torch.nn.functional.relu(h)
                _ = torch.matmul(h, w2)
            if device.type == "cuda":
                torch.cuda.synchronize()
            
            elapsed = (time.perf_counter() - t0) / iters * 1000
            throughput = bs * 1000 / elapsed
            latency = elapsed
            
            results.append({
                "batch_size": bs, "latency_ms": latency,
                "throughput": throughput,
            })
            
            print(f"   Batch={bs:3d}: latency={latency:.2f}ms, throughput={throughput:.1f} samples/s")
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"   Batch={bs:3d}: [OOM]")
            else:
                print(f"   Batch={bs:3d}: [ERR] {str(e)[:50]}")

    return results

print(f"\n--- Batch Inference Benchmark ---")
batch_results = benchmark_batch_inference()

if batch_results:
    best = max(batch_results, key=lambda r: r["throughput"])
    print(f"\n[OPTIMAL] batch_size={best['batch_size']}, throughput={best['throughput']:.1f} samples/s")

print(f"\n[ROCm Batch Best Practice]:")
print(f"   - vLLM + ROCm: --max-num-seqs for max concurrency")
print(f"   - MI300X large VRAM: use larger batch sizes")
print(f"   - Mixed precision: --dtype float16 for higher throughput")

[OPT] Batch Inference Throughput

--- Batch Inference Benchmark ---
   Batch=  1: latency=1.28ms, throughput=781.8 samples/s
   Batch=  2: latency=2.50ms, throughput=800.7 samples/s
   Batch=  4: latency=6.99ms, throughput=572.5 samples/s
   Batch=  8: latency=9.97ms, throughput=802.6 samples/s
   Batch= 16: latency=19.92ms, throughput=803.3 samples/s

[OPTIMAL] batch_size=16, throughput=803.3 samples/s

[ROCm Batch Best Practice]:
   - vLLM + ROCm: --max-num-seqs for max concurrency
   - MI300X large VRAM: use larger batch sizes
   - Mixed precision: --dtype float16 for higher throughput


In [65]:
# ============================================================
# Cell 2.6: Optimization Summary
# ============================================================

print("=" * 60)
print("[SUMMARY] Inference Optimization Overview")
print("=" * 60)

# Try to create a chart (optional)
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    
    techniques = ["FP32 Baseline", "FP16 Mixed", "INT8 Quant", "INT4 (NF4)", "+KV-Cache", "+Batch x4"]
    speedups = [1.0, 2.0, 4.0, 6.0, 12.0, 48.0]
    colors = ["#808080", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Speedup bar chart
    ax1 = axes[0]
    bars = ax1.bar(range(len(techniques)), speedups, color=colors, edgecolor='white', linewidth=0.5)
    ax1.set_xticks(range(len(techniques)))
    ax1.set_xticklabels(techniques, rotation=30, ha='right', fontsize=9)
    ax1.set_ylabel("Cumulative Speedup (x)", fontsize=12)
    ax1.set_title("ROCm Inference Optimization Stack\n(AMD GPU Cumulative Speedup)", fontsize=13, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    for bar, val in zip(bars, speedups):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.0f}x", ha='center', fontsize=10, fontweight='bold')
    
    # Subplot 2: VRAM pie chart
    ax2 = axes[1]
    mem_labels = ["Weights (FP16)", "KV-Cache", "Activations", "Free"]
    mem_sizes = [14, 2, 3, 173]
    mem_colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#E8E8E8"]
    
    ax2.pie(mem_sizes, labels=mem_labels, autopct='%1.1f%%',
            colors=mem_colors, startangle=90, textprops={'fontsize': 10})
    ax2.set_title("AMD MI300X (192GB) Memory Allocation\n7B Model FP16 Inference", fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig("roc_m_optimization_summary.png", dpi=150, bbox_inches='tight')
    print("\n[OK] Chart saved: rocm_optimization_summary.png")
except Exception as e:
    print(f"\n[WARN] Chart not created: {e}")

print(f"\n{'='*60}")
print(f"Optimization Summary Table")
print(f"{'='*60}")
print(f"   {'Technique':<20} {'Speedup':>8} {'Cumulative':>12} {'VRAM Saved':>12}")
print(f"   {'-'*54}")
print(f"   {'FP16 Mixed Prec.':<20} {'2x':>8} {'2x':>12} {'50%':>12}")
print(f"   {'INT8 Quantization':<20} {'2x':>8} {'4x':>12} {'75%':>12}")
print(f"   {'INT4 Quantization':<20} {'1.5x':>8} {'6x':>12} {'87.5%':>12}")
print(f"   {'KV-Cache':<20} {'2x':>8} {'12x':>12} {'n/a':>12}")
print(f"   {'Batch x4':<20} {'4x':>8} {'48x':>12} {'n/a':>12}")

print(f"\n[ROCm-Specific Tips]:")
print(f"   1. export HSA_OVERRIDE_GFX_VERSION=11.0.0")
print(f"   2. torch.compile() with ROCm Triton backend")
print(f"   3. MIOpen fusion: export MIOPEN_FIND_ENFORCE=3")
print(f"   4. vLLM ROCm: --enforce-eager to disable CUDA Graph (some cases)")

[SUMMARY] Inference Optimization Overview

[WARN] Chart not created: No module named 'matplotlib'

Optimization Summary Table
   Technique             Speedup   Cumulative   VRAM Saved
   ------------------------------------------------------
   FP16 Mixed Prec.           2x           2x          50%
   INT8 Quantization          2x           4x          75%
   INT4 Quantization        1.5x           6x        87.5%
   KV-Cache                   2x          12x          n/a
   Batch x4                   4x          48x          n/a

[ROCm-Specific Tips]:
   1. export HSA_OVERRIDE_GFX_VERSION=11.0.0
   2. torch.compile() with ROCm Triton backend
   3. MIOpen fusion: export MIOPEN_FIND_ENFORCE=3
   4. vLLM ROCm: --enforce-eager to disable CUDA Graph (some cases)


## Chapter 3: Agent Construction

Building on the Skill Agent framework and inference optimization from previous chapters, we now construct three complete Agent applications:

```
+-------------------------------------------------------+
|                  Agent Architecture                    |
|                                                        |
|  +----------+  +----------+  +----------------------+ |
|  |RAG Agent |  |Code Agent|  | Multi-Agent Team     | |
|  |Doc Q&A   |  |Code Gen  |  | Multi-agent collab   | |
|  +----+-----+  +----+-----+  +----------+-----------+ |
|       |             |                  |              |
|  +----+-------------+------------------+------------+ |
|  |           ROCm LLM Backend                       | |
|  |    (FP16/INT4 + KV-Cache + Batch)               | |
|  +--------------------------------------------------+ |
+-------------------------------------------------------+
```

In [76]:
# ============================================================
# Cell 3.1: RAG Agent -- Retrieval-Augmented Generation
# ============================================================

print("=" * 60)
print("[AGENT] RAG Agent: Retrieval-Augmented Generation")
print("=" * 60)

import re as _re
from typing import List, Tuple
from collections import Counter

class SimpleVectorStore:
    """Simple vector store using TF-IDF cosine similarity
    
    Production alternatives:
    - chromadb / faiss / milvus
    - sentence-transformers for embeddings
    """
    
    def __init__(self):
        self.documents: List[str] = []
        self.embeddings: List[torch.Tensor] = []
        self.vocab: Dict[str, int] = {}  # 全局词汇表
    
    def _compute_tfidf(self, text: str) -> torch.Tensor:
        """Simple TF-IDF vectorization"""
        words = _re.findall(r'\w+', text.lower())
        word_count = Counter(words)
        total = len(words) or 1
        
        vec = torch.zeros(len(self.vocab))
        for word, count in word_count.items():
            if word in self.vocab:
                tf = count / total
                df = sum(1 for doc in self.documents if word in doc.lower()) + 1
                idf = torch.log(torch.tensor((len(self.documents) + 1) / df))
                vec[self.vocab[word]] = tf * idf.item()
        
        return vec
    
    def add(self, documents: List[str]):
        """Add documents to store"""
        assert len(self.documents) == 0, "TBD: 暂时支持一次性添加"
        all_words = set()
        for doc in documents:
            all_words.update(_re.findall(r'\w+', doc.lower()))
        self.vocab = {w: i for i, w in enumerate(sorted(all_words))}

        for doc in documents:
            self.documents.append(doc)
            self.embeddings.append(self._compute_tfidf(doc))
        print(f"   [DOC] Added {len(documents)} docs, total: {len(self.documents)}")
    
    def search(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """Search most relevant documents"""
        query_vec = self._compute_tfidf(query)
        
        scores = []
        for i, doc_vec in enumerate(self.embeddings):
            if doc_vec.norm() > 0 and query_vec.norm() > 0:
                sim = torch.cosine_similarity(query_vec.unsqueeze(0), doc_vec.unsqueeze(0)).item()
            else:
                sim = 0.0
            scores.append((i, sim))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [(self.documents[i], score) for i, score in scores[:top_k]]


class RAGSkill(Skill):
    """RAG search skill"""
    
    def __init__(self, vector_store: SimpleVectorStore):
        self.store = vector_store
    
    @property
    def name(self) -> str:
        return "rag_search"
    
    @property
    def description(self) -> str:
        return "Search knowledge base for relevant document snippets"
    
    def execute(self, query: str = "", top_k: int = 3, **kwargs) -> SkillResult:
        results = self.store.search(query, top_k=top_k)
        
        if not results or results[0][1] < 0.01:
            return SkillResult(success=True, data={"query": query, "results": [], "message": "No relevant docs found"})
        
        formatted = [{"content": doc[:300], "relevance": f"{score:.4f}"} for doc, score in results]
        return SkillResult(success=True, data={"query": query, "num_results": len(formatted), "results": formatted})


# Build AMD GPU/ROCm knowledge base
print("\n--- Building AMD GPU/ROCm Knowledge Base ---")

docs = [
    "AMD ROCm is AMD's open-source GPU computing platform, competing with NVIDIA CUDA. ROCm 7.0 was released in September 2025, delivering 3.8x inference performance improvement over ROCm 6 (based on DeepSeek R1 benchmark).",
    
    "AMD Instinct MI300X uses CDNA 3 architecture, featuring 192GB HBM3 memory with 5.3 TB/s bandwidth. A single card can hold FP16 weights of a 70B parameter model without tensor parallelism.",
    
    "ROCm supports PyTorch, TensorFlow, JAX, vLLM and other major frameworks. Install PyTorch ROCm: pip install torch --index-url https://download.pytorch.org/whl/rocm7.0",
    
    "AMD MI355X is based on CDNA 4 architecture, with FP8 throughput claimed to exceed NVIDIA Blackwell B200 by 30%. Expected mass production in H1 2026.",
    
    "ROCm inference optimization: FP16/BF16 mixed precision, INT8/INT4 quantization, Flash Attention v2, Continuous Batching (vLLM), Tensor Parallelism (multi-GPU).",
    
    "ROCm 7.0 new features: HIP 6.4 runtime, MIOpen 3.0 fused kernels, RCCL communication optimization, Triton backend support.",
    
    "AMD Radeon RX 7900 XTX is a consumer GPU with 24GB VRAM, ROCm support, suitable for running 7B-13B model inference and LoRA fine-tuning locally.",
]

store = SimpleVectorStore()
store.add(docs)

# Test search
for q in ["MI300X VRAM size?", "ROCm 7.0 new features?", "How to install PyTorch on AMD GPU?"]:
    results = store.search(q, top_k=2)
    print(f"\n   [QUERY] {q}")
    for doc, score in results:
        print(f"      [{score:.4f}] {doc[:80]}...")

# Register RAG Skill
rag_skill = RAGSkill(store)
skill_manager.register(rag_skill)
print(f"\n[OK] RAG knowledge base ready: {len(store.documents)} docs, {skill_manager}")

[AGENT] RAG Agent: Retrieval-Augmented Generation

--- Building AMD GPU/ROCm Knowledge Base ---
   [DOC] Added 7 docs, total: 7

   [QUERY] MI300X VRAM size?
      [0.1699] AMD Radeon RX 7900 XTX is a consumer GPU with 24GB VRAM, ROCm support, suitable ...
      [0.1443] AMD Instinct MI300X uses CDNA 3 architecture, featuring 192GB HBM3 memory with 5...

   [QUERY] ROCm 7.0 new features?
      [0.4188] ROCm 7.0 new features: HIP 6.4 runtime, MIOpen 3.0 fused kernels, RCCL communica...
      [0.0218] ROCm supports PyTorch, TensorFlow, JAX, vLLM and other major frameworks. Install...

   [QUERY] How to install PyTorch on AMD GPU?
      [0.5612] ROCm supports PyTorch, TensorFlow, JAX, vLLM and other major frameworks. Install...
      [0.0723] AMD MI355X is based on CDNA 4 architecture, with FP8 throughput claimed to excee...
   [REG] rag_search: Search knowledge base for relevant docum...

[OK] RAG knowledge base ready: 7 docs, SkillManager(5 skills: ['calculator', 'text_processor', 'code

In [70]:
# ============================================================
# Cell 3.2: Code Agent -- Code Generation & Execution
# ============================================================

print("=" * 60)
print("[AGENT] Code Agent: Generation & Safe Execution")
print("=" * 60)

class CodeExecutionSkill(Skill):
    """Code execution skill with safety sandbox"""
    
    SAFE_BUILTINS = {
        "abs": abs, "all": all, "any": any, "bool": bool, "dict": dict,
        "enumerate": enumerate, "filter": filter, "float": float, "int": int,
        "len": len, "list": list, "map": map, "max": max, "min": min,
        "print": print, "range": range, "round": round, "set": set,
        "sorted": sorted, "str": str, "sum": sum, "tuple": tuple, "zip": zip,
    }
    
    @property
    def name(self) -> str:
        return "code_executor"
    
    @property
    def description(self) -> str:
        return "Safely execute Python code and return results"
    
    @property
    def parameters_schema(self) -> Dict:
        return {"type": "object", "properties": {"code": {"type": "string", "description": "Python code"}}, "required": ["code"]}
    
    def execute(self, code: str = "", **kwargs) -> SkillResult:
        """Execute code in restricted environment"""
        import io as _io
        from contextlib import redirect_stdout, redirect_stderr
        
        forbidden = ["__import__", "exec(", "eval(", "open(", "os.", "subprocess", "shutil", "importlib", "compile("]
        code_lower = code.lower()
        for kw in forbidden:
            if kw in code_lower:
                return SkillResult(success=False, error=f"Security: '{kw}' blocked (sandbox)")
        
        namespace = {"__builtins__": self.SAFE_BUILTINS, "__name__": "__sandbox__"}
        stdout_buf = _io.StringIO()
        stderr_buf = _io.StringIO()
        
        try:
            with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
                exec(code, namespace)
            
            output = stdout_buf.getvalue()
            errors = stderr_buf.getvalue()
            return SkillResult(success=True, data={"output": output.strip() or "(no output)", "errors": errors.strip() or None, "code": code[:200]})
        except Exception as e:
            return SkillResult(success=False, error=f"Code error: {e}")


class CodeGenerationSkill(Skill):
    """Code generation skill (template-based demo)"""
    
    @property
    def name(self) -> str:
        return "code_generator"
    
    @property
    def description(self) -> str:
        return "Generate Python code from natural language description"
    
    def execute(self, task: str = "", **kwargs) -> SkillResult:
        task_lower = task.lower()
        
        templates = {
            "fibonacci": '''def fibonacci(n):
    a, b = 0, 1
    result = []
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

print(fibonacci(10))''',
            "sort": '''def sort_and_filter(data, threshold=0):
    sorted_data = sorted(data)
    return [x for x in sorted_data if x > threshold]

test = [3, 1, 4, 1, 5, 9, 2, 6]
print(f"Original: {test}")
print(f"Processed: {sort_and_filter(test, threshold=3)}")''',
            "prime": '''def is_prime(n):
    if n < 2: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0: return False
    return True

primes = [n for n in range(2, 100) if is_prime(n)]
print(f"Primes < 100: {primes}")
print(f"Count: {len(primes)}")'''
        }
        
        for key, code in templates.items():
            if key in task_lower:
                return SkillResult(success=True, data={"task": task, "code": code.strip(), "language": "python"})
        
        default_code = """data = [1, 2, 3, 4, 5]
result = sum(data)
print(f"Input: {data}")
print(f"Sum: {result}")"""
        return SkillResult(success=True, data={"task": task, "code": default_code.strip(), "language": "python"})


# Register Code Skills
code_exec = CodeExecutionSkill()
code_gen = CodeGenerationSkill()
skill_manager.register(code_exec)
skill_manager.register(code_gen)

print(f"\n[INFO] {skill_manager}")

# End-to-end test: generate -> execute
print(f"\n--- End-to-End Test: Generate + Execute ---")
gen_result = code_gen(task="fibonacci")
print(f"[GEN] Code:\n{gen_result.data['code'][:200]}...")

exec_result = code_exec(code=gen_result.data["code"])
print(f"\n[EXEC] Result: {exec_result.data['output']}")

# Security sandbox test
print(f"\n--- Security Sandbox Test ---")
for bad_code in ["import os\nprint(os.listdir('.'))", "open('test.txt', 'w').write('hello')"]:
    result = code_exec(code=bad_code)
    status = "[BLOCKED]" if not result.success else "[PASS]"
    print(f"   {status}: {bad_code[:50]}...")

[AGENT] Code Agent: Generation & Safe Execution
   [REG] code_executor: Safely execute Python code and return re...
   [REG] code_generator: Generate Python code from natural langua...

[INFO] SkillManager(4 skills: ['calculator', 'text_processor', 'code_executor', 'code_generator'])

--- End-to-End Test: Generate + Execute ---
[GEN] Code:
def fibonacci(n):
    a, b = 0, 1
    result = []
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

print(fibonacci(10))...

[EXEC] Result: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

--- Security Sandbox Test ---
   [BLOCKED]: import os
print(os.listdir('.'))...
   [BLOCKED]: open('test.txt', 'w').write('hello')...


In [77]:
# ============================================================
# Cell 3.3: Multi-Agent Collaboration System
# ============================================================

print("=" * 60)
print("[AGENT] Multi-Agent Collaboration System")
print("=" * 60)

from enum import Enum

class AgentRole(Enum):
    ORCHESTRATOR = "orchestrator"
    RESEARCHER = "researcher"
    CODER = "coder"
    REVIEWER = "reviewer"

@dataclass
class AgentMessage:
    """Inter-agent message"""
    sender: str
    receiver: str
    content: str
    msg_type: str = "info"

class SpecialistAgent:
    """Specialized agent with domain-specific skills"""
    
    def __init__(self, name: str, role: AgentRole, skills: List[Skill]):
        self.name = name
        self.role = role
        self.skills = skills
        self.inbox: List[AgentMessage] = []
    
    def receive(self, msg: AgentMessage):
        self.inbox.append(msg)
        print(f"   [MSG] [{self.name}] <- [{msg.sender}]: {msg.content[:60]}...")
    
    def process(self, task: str) -> SkillResult:
        """Try all skills, return first success"""
        for skill in self.skills:
            if skill.name == "rag_search":
                result = skill(query=task)
            elif skill.name == "code_executor":
                result = skill(code=task)
            elif skill.name == "calculator":
                result = skill(expression=task)
            elif skill.name == "text_processor":
                result = skill(text=task, operation="stats")
            else:
                continue
            
            if result.success:
                print(f"   [OK] [{self.name}] completed via {skill.name}")
                return result
        
        return SkillResult(success=False, error=f"[{self.name}] Cannot handle: {task}")


class Orchestrator:
    """Task decomposition and agent scheduling"""
    
    def __init__(self):
        self.agents: Dict[str, SpecialistAgent] = {}
        self.task_log: List[Dict] = []
    
    def register(self, agent: SpecialistAgent):
        self.agents[agent.name] = agent
        print(f"   [REG] Agent: [{agent.role.value}] {agent.name}")
    
    def run_pipeline(self, task: str, pipeline: List[str]) -> Dict:
        """Execute task through agent pipeline"""
        
        print(f"\n{'='*40}")
        print(f"[TASK] {task}")
        print(f"[PIPE] {' -> '.join(pipeline)}")
        print(f"{'='*40}")
        
        results = {}
        context = {"task": task}
        
        for step_idx, agent_name in enumerate(pipeline):
            agent = self.agents.get(agent_name)
            if not agent:
                results[agent_name] = {"error": f"Agent '{agent_name}' not found"}
                continue
            
            print(f"\n   [STEP {step_idx+1}] [{agent_name}] processing...")
            
            if agent.role == AgentRole.RESEARCHER:
                step_task = f"Search: {task}"
                result = agent.process(step_task)
                context["research"] = result.data if result.success else {}
            elif agent.role == AgentRole.CODER:
                result = agent.process(task)
                context["code"] = result.data if result.success else {}
            elif agent.role == AgentRole.REVIEWER:
                review = self._review(context)
                results[agent_name] = review
                continue
            
            results[agent_name] = {
                "success": result.success,
                "data": result.data if result.success else None,
                "error": result.error if not result.success else None,
            }
        
        summary = {
            "task": task, "pipeline": pipeline, "results": results,
            "all_success": all(r.get("success", True) for r in results.values()),
        }
        self.task_log.append(summary)
        
        print(f"\n{'='*40}")
        print(f"[RESULT] {'[OK] All success' if summary['all_success'] else '[WARN] Partial failure'}")
        
        return summary
    
    def _review(self, context: Dict) -> Dict:
        issues = []
        if "code" in context and context["code"]:
            code_out = context["code"].get("output", "")
            if "error" in code_out.lower():
                issues.append("Code contains errors")
        if "research" in context and context["research"]:
            if context["research"].get("num_results", 0) == 0:
                issues.append("No docs retrieved")
        return {"success": len(issues) == 0, "data": {"issues": issues, "passed": len(issues) == 0}}
    
    def stats(self) -> Dict:
        return {
            "num_agents": len(self.agents),
            "num_tasks": len(self.task_log),
            "success_rate": sum(1 for t in self.task_log if t["all_success"]) / max(len(self.task_log), 1),
        }


# Build Multi-Agent system
print("\n--- Building Multi-Agent System ---")

researcher = SpecialistAgent("ResearcherBot", AgentRole.RESEARCHER,
    [rag_skill, skill_manager.get("text_processor")])
coder = SpecialistAgent("CoderBot", AgentRole.CODER,
    [code_exec, code_gen, skill_manager.get("calculator")])
reviewer = SpecialistAgent("ReviewerBot", AgentRole.REVIEWER, [])

orch = Orchestrator()
orch.register(researcher)
orch.register(coder)
orch.register(reviewer)

print(f"\n[INFO] System status: {orch.stats()}")

# Test Pipelines
print("\n" + "=" * 60)
print("[TEST] Multi-Agent Pipeline Tests")
print("=" * 60)

orch.run_pipeline(task="MI300X VRAM and architecture features", pipeline=["ResearcherBot", "ReviewerBot"])
orch.run_pipeline(task="fibonacci", pipeline=["CoderBot", "ReviewerBot"])

print(f"\n[FINAL] Stats: {json.dumps(orch.stats(), indent=2, ensure_ascii=False)}")

[AGENT] Multi-Agent Collaboration System

--- Building Multi-Agent System ---
   [REG] Agent: [researcher] ResearcherBot
   [REG] Agent: [coder] CoderBot
   [REG] Agent: [reviewer] ReviewerBot

[INFO] System status: {'num_agents': 3, 'num_tasks': 0, 'success_rate': 0.0}

[TEST] Multi-Agent Pipeline Tests

[TASK] MI300X VRAM and architecture features
[PIPE] ResearcherBot -> ReviewerBot

   [STEP 1] [ResearcherBot] processing...
   [OK] [ResearcherBot] completed via rag_search

   [STEP 2] [ReviewerBot] processing...

[RESULT] [OK] All success

[TASK] fibonacci
[PIPE] CoderBot -> ReviewerBot

   [STEP 1] [CoderBot] processing...

   [STEP 2] [ReviewerBot] processing...

[RESULT] [WARN] Partial failure

[FINAL] Stats: {
  "num_agents": 3,
  "num_tasks": 2,
  "success_rate": 0.5
}


In [78]:
# ============================================================
# Cell 3.4: Full Agent Integration Test
# ============================================================

print("=" * 60)
print("[TEST] Full Agent Integration")
print("=" * 60)

class IntegratedAgent:
    """Integrated Agent: RAG + Code + Multi-Agent combined"""
    
    def __init__(self, skill_mgr: SkillManager, orchestrator: Orchestrator):
        self.skills = skill_mgr
        self.orch = orchestrator
    
    def solve(self, user_query: str) -> str:
        """Smart routing: dispatch to appropriate pipeline based on query type"""
        
        query_lower = user_query.lower()
        
        if any(w in query_lower for w in ["amd", "roc", "gpu", "mi300", "mi355", "vram", "architect"]):
            pipeline = ["ResearcherBot", "ReviewerBot"]
            intent = "RAG Knowledge Retrieval"
        elif any(w in query_lower for w in ["code", "calc", "fibonacci", "sort", "prime"]):
            pipeline = ["CoderBot", "ReviewerBot"]
            intent = "Code Generation & Execution"
        elif any(w in query_lower for w in ["full", "pipeline", "all"]):
            pipeline = ["ResearcherBot", "CoderBot", "ReviewerBot"]
            intent = "Full Pipeline"
        else:
            pipeline = ["ResearcherBot"]
            intent = "Default RAG"
        
        print(f"\n[INTENT] {intent}")
        print(f"[PIPE] {' -> '.join(pipeline)}")
        
        result = self.orch.run_pipeline(user_query, pipeline)
        
        if result["all_success"]:
            return f"[OK] Task complete ({intent})"
        else:
            failed = [k for k, v in result["results"].items() if not v.get("success", True)]
            return f"[WARN] Partial failure ({intent}), failed: {failed}"
    
    def interactive_demo(self):
        """Demo with test queries"""
        queries = [
            "AMD MI300X VRAM size and architecture?",
            "Calculate fibonacci first 10 numbers",
            "ROCm 7.0 inference optimization improvements?",
        ]
        
        for i, query in enumerate(queries, 1):
            print(f"\n{'#'*60}")
            print(f"# Demo {i}/{len(queries)}")
            print(f"{'#'*60}")
            reply = self.solve(query)
            print(f"\n[REPLY] {reply}")


# Run integration test
integrated_agent = IntegratedAgent(skill_manager, orch)
integrated_agent.interactive_demo()

print(f"\n{'='*60}")
print(f"[OK] All Agent integration tests complete!")
print(f"   Registered skills: {len(skill_manager)}")
print(f"   Registered agents: {orch.stats()['num_agents']}")
print(f"   Success rate: {orch.stats()['success_rate']*100:.0f}%")
print(f"{'='*60}")

[TEST] Full Agent Integration

############################################################
# Demo 1/3
############################################################

[INTENT] RAG Knowledge Retrieval
[PIPE] ResearcherBot -> ReviewerBot

[TASK] AMD MI300X VRAM size and architecture?
[PIPE] ResearcherBot -> ReviewerBot

   [STEP 1] [ResearcherBot] processing...
   [OK] [ResearcherBot] completed via rag_search

   [STEP 2] [ReviewerBot] processing...

[RESULT] [OK] All success

[REPLY] [OK] Task complete (RAG Knowledge Retrieval)

############################################################
# Demo 2/3
############################################################

[INTENT] Code Generation & Execution
[PIPE] CoderBot -> ReviewerBot

[TASK] Calculate fibonacci first 10 numbers
[PIPE] CoderBot -> ReviewerBot

   [STEP 1] [CoderBot] processing...

   [STEP 2] [ReviewerBot] processing...

[RESULT] [WARN] Partial failure

[REPLY] [WARN] Partial failure (Code Generation & Execution), failed: ['Coder

## Chapter 4: Summary & Next Steps

### Achievements

| Module | Content | Status |
|--------|---------|--------|
| **Environment** | ROCm + PyTorch install & verification | [OK] |
| **Skill Agent** | Skill base class, manager, LLM backend | [OK] |
| **Inference Opt** | FP16/INT8/INT4/KV-Cache/Batch | [OK] |
| **RAG Agent** | Vector store, retrieval-augmented gen | [OK] |
| **Code Agent** | Code generation, safe sandbox exec | [OK] |
| **Multi-Agent** | Orchestrator + multi-role collaboration | [OK] |

### ROCm Ecosystem Key Tools

```bash
# Core framework
pip install torch --index-url https://download.pytorch.org/whl/rocm7.0
pip install transformers accelerate

# Quantization
pip install bitsandbytes  # ROCm 5.4+

# Inference engine
pip install vllm
# vllm serve Qwen/Qwen2.5-7B-Instruct --dtype float16

# Fine-tuning
pip install trl peft  # LoRA/QLoRA
```

### Further Reading

- [ROCm Official Docs](https://rocm.docs.amd.com)
- [PyTorch on AMD GPU](https://pytorch.org/get-started/locally/)
- [vLLM ROCm Support](https://docs.vllm.ai/en/latest/getting_started/amd-installation.html)
- [HuggingFace + AMD](https://huggingface.co/docs/transformers/main/en/perf_amd_gpu)

### Next Steps

1. **Deploy**: Run this notebook on a real AMD GPU
2. **Profile**: Use `rocprof` to analyze kernel bottlenecks
3. **Scale**: Configure PyTorch FSDP or DeepSpeed ZeRO for multi-GPU
4. **Production**: Deploy agent as vLLM API service

---
*Notebook based on ROCm 7.0 + PyTorch 2.x | All code directly migratable to real AMD GPU*

In [79]:
# ============================================================
# Cell 4.1: Final Self-Check -- Validate All Modules
# ============================================================

print("=" * 60)
print("[CHECK] Final Self-Check")
print("=" * 60)

checks = []

# 1. Skill Manager
try:
    assert len(skill_manager) >= 5, f"Insufficient skills: {len(skill_manager)}"
    checks.append(("[OK]", "Skill Manager", f"{len(skill_manager)} skills registered"))
except Exception as e:
    checks.append(("[ERR]", "Skill Manager", str(e)))

# 2. Skill Execution
try:
    result = skill_manager.execute("calculator", expression="1+1")
    assert result.success and result.data["result"] == 2
    checks.append(("[OK]", "Calculator Skill", "1+1=2 test passed"))
except Exception as e:
    checks.append(("[ERR]", "Calculator Skill", str(e)))

# 3. RAG Search
try:
    result = skill_manager.execute("rag_search", query="MI300X")
    assert result.success
    checks.append(("[OK]", "RAG Search", f"Retrieved {result.data.get('num_results', 0)} results"))
except Exception as e:
    checks.append(("[ERR]", "RAG Search", str(e)))

# 4. Code Execution
try:
    result = skill_manager.execute("code_executor", code="print(sum([1,2,3,4,5]))")
    assert result.success
    checks.append(("[OK]", "Code Executor", "Sandbox execution OK"))
except Exception as e:
    checks.append(("[ERR]", "Code Executor", str(e)))

# 5. Multi-Agent Pipeline
try:
    assert orch.stats()["num_agents"] == 3
    checks.append(("[OK]", "Multi-Agent", f"{orch.stats()['num_agents']} agents ready"))
except Exception as e:
    checks.append(("[ERR]", "Multi-Agent", str(e)))

# 6. LLM Backend
try:
    response = llm.generate("test")
    assert len(response) > 0
    checks.append(("[OK]", "LLM Backend", f"Generated {len(response)} chars"))
except Exception as e:
    checks.append(("[ERR]", "LLM Backend", str(e)))

# 7. ROCm/PyTorch
try:
    import torch
    rocm_available = torch.cuda.is_available()
    status = "GPU available" if rocm_available else "CPU mode"
    checks.append(("[OK]" if rocm_available else "[--]", "PyTorch Device", status))
except Exception as e:
    checks.append(("[ERR]", "PyTorch Device", str(e)))

# Output
print(f"\n{'Status':<8} {'Module':<22} {'Detail':<40}")
print(f"{'-'*72}")
for status, module, detail in checks:
    print(f"{status:<8} {module:<22} {detail:<40}")
print(f"{'-'*72}")
passed = sum(1 for s, _, _ in checks if s in ("[OK]", "[--]"))
print(f"\nTotal: {passed}/{len(checks)} passed")

if passed == len(checks):
    print("\n[SUCCESS] All checks passed! Notebook execution complete!")
else:
    print(f"\n[WARN] {len(checks) - passed} items failed. Check modules above.")

print("=" * 60)

[CHECK] Final Self-Check

Status   Module                 Detail                                  
------------------------------------------------------------------------
[OK]     Skill Manager          5 skills registered                     
[OK]     Calculator Skill       1+1=2 test passed                       
[OK]     RAG Search             Retrieved 3 results                     
[OK]     Code Executor          Sandbox execution OK                    
[OK]     Multi-Agent            3 agents ready                          
[OK]     LLM Backend            Generated 132 chars                     
[OK]     PyTorch Device         GPU available                           
------------------------------------------------------------------------

Total: 7/7 passed

[SUCCESS] All checks passed! Notebook execution complete!
